# 03. 기준선 모델

1~7일 뒤의 일별 판매량을 예측하기 전에, 각 목표일과 같은 요일의 직전 주 판매량이 어느 정도 성능을 내는지 확인합니다. 이 값은 모든 예측 거리에서 기준일 이전에 관측되므로 안전하게 사용할 수 있습니다.

## 패키지 준비

Seaborn과 나눔고딕 글꼴을 설치합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib


## Library


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import koreanize_matplotlib


## Utils


In [ ]:
def rmsle(actual, predicted):
    actual = np.clip(np.asarray(actual), 0, None)
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))


def mae(actual, predicted):
    return np.mean(np.abs(np.asarray(actual) - np.asarray(predicted)))


def evaluate_baselines(frame, columns):
    rows = []
    for name, column in columns.items():
        rows.append({"model": name, "rmsle": rmsle(frame["target_sales"], frame[column]), "mae": mae(frame["target_sales"], frame[column])})
    return pd.DataFrame(rows).set_index("model").sort_values("rmsle")

## 실행 설정


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw"
OUTPUT_PATH = PROJECT_ROOT / "outputs"

## Baseline 선정 근거

> **EDA 근거:** 전체 판매량의 7일 자기상관(0.826)이 1일(0.767)보다 높고, 추세를 제거한 뒤에도 7일 주기가 남았습니다. 따라서 같은 요일의 과거 판매량을 가장 단순한 비교 기준으로 둡니다. 자세한 분석은 `01_eda.ipynb`에 있습니다.

목표일을 `d = t+h` (`h=1~7`)라고 할 때 두 Baseline은 다음과 같습니다.

| Baseline | 예측값 | 채택 이유 |
|---|---|---|
| 전주 동일 요일 | 같은 매장·상품군의 `d−7` 판매량 | 가장 최근의 주간 패턴을 그대로 반영 |
| 최근 4주 동일 요일 평균 | `d−7`, `d−14`, `d−21`, `d−28` 판매량의 평균 | 특정 한 주의 급증·급감 영향을 완화 |

`h≤7`이므로 사용한 날짜는 모두 기준일 `t` 또는 그 이전입니다. 두 기준선은 주간 패턴만으로 얻을 수 있는 성능을 보여주며, 아래 LightGBM 예측이 이를 개선하는지 확인합니다.


In [ ]:
predictions_path = OUTPUT_PATH / "validation_predictions_lightgbm.csv"
predictions_path = predictions_path if predictions_path.exists() else OUTPUT_PATH / "validation_predictions.csv"
predictions = pd.read_csv(
    predictions_path,
    parse_dates=["date", "target_date"],
)

In [ ]:
baseline_columns = {
    "LightGBM": "prediction",
    "Previous-week daily profile": "sales_same_weekday_last_week",
    "Same-weekday 4-week mean": "sales_same_weekday_4w_mean",
}
baseline_scores = evaluate_baselines(predictions, baseline_columns)
baseline_scores.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
baseline_scores["rmsle"].sort_values().plot.barh(ax=axes[0], title="RMSLE")
baseline_scores["mae"].sort_values().plot.barh(ax=axes[1], color="tab:orange", title="MAE")
axes[0].set(xlabel="lower is better", ylabel="")
axes[1].set(xlabel="lower is better", ylabel="")
plt.tight_layout()

전체 성능뿐 아니라 `forecast_horizon`별 RMSLE와 MAE를 함께 확인합니다. 예측 거리가 길어질수록 오차가 증가하는지, 통합 LightGBM이 각 거리에서 주간 계절 기준선을 안정적으로 개선하는지가 핵심 판단 기준입니다.

In [ ]:
horizon_scores = []
for horizon, group in predictions.groupby("forecast_horizon"):
    for name, column in baseline_columns.items():
        horizon_scores.append({
            "forecast_horizon": horizon,
            "model": name,
            "rmsle": rmsle(group["target_sales"], group[column]),
            "mae": mae(group["target_sales"], group[column]),
        })
horizon_scores = pd.DataFrame(horizon_scores)
horizon_scores.pivot(index="forecast_horizon", columns="model", values=["rmsle", "mae"]).round(4)